# Reddit Logs to YTMusic Playlists

In [8]:
# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'

# Path to reddit files
reddit_log_path = '..\\..\\reddit_scraper\\logs\\'
reddit_scrape_path = 'B:\\Music_Reddit_Scrape\\'
trash_path = 'B:\\Music_Reddit_Scrape_Trash\\'

In [9]:
import os
import glob
import time
import math
import glob

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

In [10]:
for folder in glob.glob(reddit_scrape_path + '*\\'):
    print(folder)


B:\Music_Reddit_Scrape\albums\
B:\Music_Reddit_Scrape\collections\
B:\Music_Reddit_Scrape\mixes_performances\
B:\Music_Reddit_Scrape\tracks\


In [11]:
def scrub_title(title):
    title = title.lower().strip()

    # remove leading numbers
    title = re.sub('^\d+', '', title).strip()
    if title.startswith('. ') or title.startswith('- '):
        title = title[2:]

    title = unicodedata.normalize('NFKD', title).encode(
        'ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    for k in [
        '(official', '[official', 'official video', '[official', '[unoffical', '[free', '(free',
        '(explicit', '[video', '(video', '(music', '(nsfw', '(unoffical', '(live', '[live',
        '(video', 'music video', 'live video', '(original', '(lyric', 'lyric video', 
        '[full', '(full', '(album)', 'album stream', 'album review', '[thissongissick',
        '(prod', 'prod.', 'produced by', '[leak', '(from', '(lofi hip', '(remastered']:
        if k in title:
            title = title.split(k)[0]
    for k in ['[hd]', '[hq]', 'hd', 'hq']:
        if k in title:
            title=title.replace(k, '')
    return title

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## YTMusic API and Functions

In [12]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))
yt_res_cache = {}
yt_unmatched_cache = {}

## Parse Reddit Scraped .mp3s and query YTMusic for Match

* Just tracks so far

#### Last run 10/23/2021


### Search ytmusic songs for scraped Reddit tracks

In [ ]:

matched_entries = []
unmatched_entries = []
subreddit_dir = os.path.join(reddit_scrape_path, 'tracks')
for sub_dir in os.listdir(subreddit_dir):
    sub, agg, count = sub_dir.split('_')
    for f in os.listdir(os.path.join(subreddit_dir, sub_dir)):
        filepath = os.path.join(subreddit_dir, sub_dir, f)
        fname, ext = os.path.splitext(f)
        if not fname:
            print(f'Skipping blank file: {filepath}')
            continue
        title_key = scrub_title(fname)
        title_key = title_key.replace('-', '')
        if not title_key:
            print(f'Blank file: {title_key}, using fname: {fname}')
            title_key = fname
            continue     
         
        # Check cache for saved YTMusic query response or previous match failures
        if fname in yt_res_cache:
            match = yt_res_cache[fname]
            if 'ytmusic_artist' not in match:
                try:
                    alb_res = ytm.get_album(match['ytmusic_albumId'])
                    if 'artists' in alb_res:
                        match['ytmusic_artist'] = alb_res['artists'][0].get('name', '')
                    match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')}".lower()
                    match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
                    yt_res_cache[fname] = match
                except Exception as e:
                    print(e)

        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            unmatched_entries.append(yt_unmatched_cache[title_key])
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                res = ytm.search(query=title_key, filter='songs', limit=1)
                # Store unmatched ytmusic query
                if len(res) == 0:
                    unmatch = {}
                    unmatch['reddit_file'] = filepath
                    unmatch['reddit_source_file'] = f
                    unmatch['reddit_key'] = title_key
                    unmatch['reddit_sub'] = sub
                    unmatch['reddit_sub_id'] = f'{sub}-{title_key}'
                    unmatch['reddit_aggregator'] = agg
                    unmatched_entries.append(unmatch)
                    yt_unmatched_cache[title_key] = unmatch
                    print(f'Skipping : {title_key}')
                    continue
                else:
                    res = res[0]
                    match['is_album'] = False
                    match['ytmusic_album'] = ''
                    if 'album' in res:
                        match['ytmusic_album'] = res['album'].get('name', '')
                        match['ytmusic_albumId'] = res['album'].get('id', '')
                    if 'artists' in res:
                        match['ytmusic_artist'] = res['artists'][0].get('name', '')
                        match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                    match['ytmusic_title'] = res.get('title', '')
                    match['ytmusic_videoId'] = res.get('videoId', '')
                    match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')}".lower()
                    match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
                    match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_videoId','')}"
                    match['ytmusic_duration'] = res['duration']
                    match['ytmusic_year'] = res['year']
                    match['ytmusic_resultType'] = res['resultType']
                    yt_res_cache[fname] = match
            except Exception as e:
                print(f'Error with {title_key}: {e}')
                pass

        # Reddit fields
        match['reddit_file'] = filepath
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = fname

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Thresholds based on first round of manual quality check
            # see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
            # see: reddit_scraper\logs\ytmusic\*.png
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 70:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 90:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {fname}.\n  Match Error: {e}')

        # Update match results
        matched_entries.append(match)

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='ytmusic_playlist_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {reddit_log_path}')
match_file = os.path.join(reddit_log_path, 'ytmusic',
                          f'reddit_scraped-2019_ytmusic_scored_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_scraped-2019_ytmusic_failed_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)

## Delete files after manually grading

In [ ]:

graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_ytmusic_scored_matches_and_partly_graded_2021-10-23.tsv')

graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']

# these files can be deleted as a ytmusic match was found
can_delete = frozenset(passing.reddit_key)
print(f'Found {len(can_delete)} matches that can now be deleted from: {reddit_scrape_path}')

subreddit_dir = os.path.join(reddit_scrape_path, 'tracks')
for sub_dir in os.listdir(subreddit_dir):
    sub, agg, count = sub_dir.split('_')
    for f in os.listdir(os.path.join(subreddit_dir, sub_dir)):
        filepath = os.path.join(subreddit_dir, sub_dir, f)
        fname, ext = os.path.splitext(f)
        if not fname:
            print(f'Skipping blank file: {filepath}')
            continue
        title_key = scrub_title(fname)
        title_key = title_key.replace('-', '')
        if title_key in can_delete:
            trash_file = filepath.replace(reddit_scrape_path, trash_path)
            dest_dir = os.path.dirname(trash_file)
            if not os.path.isdir(dest_dir):
                os.makedirs(dest_dir)
            os.rename(filepath,trash_file)


## Delete manually graded round 2

In [15]:

graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_and_2021-refresh_ytmusic_scored_and_graded_matches_2021-10-24.tsv')

graded_matches_all = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches_all.loc[~graded_matches_all.reddit_file.isna()]
# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']

# these files can be deleted as a ytmusic match was found
can_delete = frozenset(passing.reddit_key)
print(f'Found {len(can_delete)} matches that can now be deleted from: {reddit_scrape_path}')

subreddit_dir = os.path.join(reddit_scrape_path, 'tracks')
for sub_dir in os.listdir(subreddit_dir):
    sub, agg, count = sub_dir.split('_')
    for f in os.listdir(os.path.join(subreddit_dir, sub_dir)):
        filepath = os.path.join(subreddit_dir, sub_dir, f)
        fname, ext = os.path.splitext(f)
        if not fname:
            print(f'Skipping blank file: {filepath}')
            continue
        title_key = scrub_title(fname)
        title_key = title_key.replace('-', '')
        if title_key in can_delete:
            trash_file = filepath.replace(reddit_scrape_path, trash_path)
            dest_dir = os.path.dirname(trash_file)
            if not os.path.isdir(dest_dir):
                os.makedirs(dest_dir)
            os.rename(filepath,trash_file)


Found 232 matches that can now be deleted from: B:\Music_Reddit_Scrape\


# TODO   Search ytmusic albums for scraped Reddit albums

In [ ]:

matched_alb_entries = []
unmatched_alb_entries = []
subreddit_dir = os.path.join(reddit_scrape_path, 'albums')
for sub_dir in os.listdir(subreddit_dir):
    sub, agg, count = sub_dir.split('_')
    for f in os.listdir(os.path.join(subreddit_dir, sub_dir)):
        filepath = os.path.join(subreddit_dir, sub_dir, f)
        fname, ext = os.path.splitext(f)
        if not fname:
            print(f'Skipping blank file: {filepath}')
            continue
        title_key = scrub_title(fname)
        title_key = title_key.replace('-', '')
        if not title_key:
            print(f'Blank file: {title_key}, using fname: {fname}')
            continue     
         
        # Check cache for saved YTMusic query response or previous match failures
        if fname in yt_res_cache:
            match = yt_res_cache[fname]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            unmatched_alb_entries.append(yt_unmatched_cache[title_key])
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                res = ytm.search(query=title_key, filter='songs', limit=1)
                # Store unmatched ytmusic query
                if len(res) == 0:
                    unmatch = {}
                    unmatch['reddit_file'] = filepath
                    unmatch['reddit_source_file'] = f
                    unmatch['reddit_key'] = title_key
                    unmatch['reddit_sub'] = sub
                    unmatch['reddit_sub_id'] = f'{sub}-{title_key}'
                    unmatch['reddit_aggregator'] = agg
                    unmatched_alb_entries.append(unmatch)
                    yt_unmatched_cache[title_key] = unmatch
                    print(f'Skipping : {title_key}')
                    continue
                else:
                    res = res[0]
                    match['is_album'] = True
                    match['ytmusic_title'] = ''
                    match['ytmusic_album'] = res.get('title', '')
                    match['ytmusic_albumId'] = res.get('browseId', '')
                    if 'artist' in res:
                        match['ytmusic_artists'] = res['artists'][0].get('name', '')
                        match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                    match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_album', '')}".lower()
                    match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_albumId','')}"
                    match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
                    match['ytmusic_duration'] = res['duration']
                    match['ytmusic_year'] = res['year']
                    match['ytmusic_resultType'] = res['resultType']
                    yt_res_cache[fname] = match
            except Exception as e:
                print(f'Error with {title_key}: {e}')
                pass

        # Reddit fields
        match['reddit_file'] = filepath
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = fname

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Thresholds based on first round of manual quality check
            # see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
            # see: reddit_scraper\logs\ytmusic\*.png
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 70:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 90:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {fname}.\n  Match Error: {e}')

        # Update match results
        matched_alb_entries.append(match)

# Process Matched entries
match_df = pd.DataFrame(matched_alb_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='ytmusic_playlist_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match album duplicates')
print(f'\n\nSaving output tsvs to {reddit_log_path}')
match_file = os.path.join(reddit_log_path, 'ytmusic',
                          f'reddit_scraped-2019_ytmusic_scored_album_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_alb_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch album duplicates')
unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_scraped-2019_ytmusic_failed_album_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)

## Create YTMusic Playlists from matches

* manually marked each match as 'ok' (pass) or 'x' (fail)
    * see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
* computed match score using a few fuzz algos
    * set basic threshold to try to automate grading

The cells below:
1. Load the graded matches and split into pass / fail matches
2. Merges the failing matches with the unmatched reddit posts, then saves a new tsv with all unmatched 
3. Save the passing tracks to respective subreddit track playlists (split into like / unrated)
4. Save the passing albums to respective subreddit album playlists (split out liked tracks to merge with track playlist)

### Done
* use fuzzy id scoring to determine if match is passing
    * if failing add to unmatched entries
    * if passing add to ytmusic {r.subreddit} playlist
        * have one playlist for albums and another for tracks
* split out 'likes' for each {r.subreddit} ytmusic playlist
* handle passing tacks and albums by adding to subreddit playlist
* handle failed matches, att them to the unmaatched tsv with their scores and extra cols intact
    * maybe retry search? look and see if search query can be done better
    * if source is youtube add youtube link to correct ytmusic sub playlist (last resort)
        * add to albums vs track playist based on duration? 
#### Last run 10/17/2021

In [6]:
# graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
#                     'reddit_ytmusic_scored_matches_and_graded_2021-10-16.tsv')
# graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)

graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_scraped-2019_ytmusic_scored_matches_and_partly_graded_2021-10-23.tsv')

graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)


# Drop duplicate entries (not needed in future tsvs, added dupe deleting in above cell)
graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
# failing = graded_matches.loc[graded_matches.manual_check == 'x']

## SKIPPED BECAUE PARTIAL Merge failing with unmatched and save new tsv


In [ ]:
# unmatched_tsv = os.path.join(reddit_log_path, 'ytmusic',
#                     'reddit_ytmusic_failed_matches_2021-10-16.tsv')
# unmatched_df = pd.read_csv(unmatched_tsv, sep='\t', index_col=0)
# all_unmatched = pd.concat([failing, unmatched_df]).sort_values('reddit_sub')
# print(f'Saving {len(all_unmatched)} unmatched reddit entries')
# all_unmatch_file = os.path.join(
#     reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_graded_{date.today()}.tsv')
# all_unmatched.to_csv(all_unmatch_file, sep='\t', header=True)

## Subreddit playlists for passing tracks

In [7]:
N=5
LIMIT = 1000
passing_tracks = passing.loc[passing.is_album == False]
completed = []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'futuregarage', 'hiphop', 'hiphop101', 'hiphopheads', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'nudisco', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic']
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing tracks')
    vids = df.ytmusic_videoId.unique().tolist()
    title=f'x_r.{sub}_scraped_tracks'
    desc = f'Matched {len(vids)} tracks from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} tracks playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)
    
    # Create Unrated Radio subset
    unrated_tracks = tracks.loc[tracks['likeStatus'] != 'LIKE']
    vids = unrated_tracks.videoId.unique().tolist()
    desc = f'Unrated radio subset of {len(vids)} entries from: {desc}'
    radio_pl_id = ytm.create_playlist(title=f'{title}_radio', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} INDIFFERENT r.{sub} radio tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Delete original playlist now thet like/unrated is split
    ytm.delete_playlist(pl_id)
    print(f'Deleted r/{sub} tracks playlist with id: {pl_id}')
    completed.append(sub)
    



Generating r.90shiphop ytmusic playlist for 484 passing tracks
Saved 484 r.90shiphop tracks playlist with id: PLWptjpDqazOwIAGi7pfVoPIBT6WCIeGrc, waiting 5 seconds...
Filtered 95 LIKE r.90shiphop tracks playlist with id: PLWptjpDqazOzr7wTWs3y4CLgRtfuyzLwm, waiting 5 seconds...
Filtered 389 INDIFFERENT r.90shiphop radio tracks playlist with id: PLWptjpDqazOzr7wTWs3y4CLgRtfuyzLwm, waiting 5 seconds...
Deleted r/90shiphop tracks playlist with id: PLWptjpDqazOwIAGi7pfVoPIBT6WCIeGrc

Generating r.blues ytmusic playlist for 462 passing tracks
Saved 462 r.blues tracks playlist with id: PLWptjpDqazOwJOACur9J1vx8SYd9QW6uQ, waiting 5 seconds...
Filtered 29 LIKE r.blues tracks playlist with id: PLWptjpDqazOwnB3lOdbWH94Dbyi-Z_M36, waiting 5 seconds...
Filtered 433 INDIFFERENT r.blues radio tracks playlist with id: PLWptjpDqazOwnB3lOdbWH94Dbyi-Z_M36, waiting 5 seconds...
Deleted r/blues tracks playlist with id: PLWptjpDqazOwJOACur9J1vx8SYd9QW6uQ

Generating r.chillmusic ytmusic playlist for 898 pa

## Subreddit playlists for passing albums

In [ ]:
SLEEP_TIME=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == True]
completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']

for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing albums')
    vids = set()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
            vids.add(track['videoId'])
    vids = list(vids)

    title=f'x_r.{sub}_albums'
    desc = f'Matched {len(vids)} albums from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} albums playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_tracks_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r.{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(SLEEP_TIME)
    completed.append(sub)

